# NB-11-02 — Retrieval, Prompting and Evaluation

This workbench accompanies L11-04, L11-05 and L11-07. It builds the RAG assistant over the
24-document handbook, compares retrievers, tests two prompt templates, and scores the results
at every stage.

In [1]:
# Make the module's reference code importable from the notebooks/ directory.
import sys, pathlib
CODE_DIR = None
for _base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_base / "code").is_dir() and (_base / "notebooks").is_dir():
        CODE_DIR = _base / "code"
        break
if CODE_DIR is None:
    CODE_DIR = pathlib.Path.cwd().parent / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print("code on path:", CODE_DIR)


code on path: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M11-generative-ai-and-large-language-models\code


In [2]:
import sys
sys.path.insert(0, ".")

import numpy as np
from gen_data import QA, UNANSWERABLE, HANDBOOK
from rag import RAGSystem, evaluate_retrievers, build_context

system = RAGSystem()
print("documents:", len(system.docs), "chunks:", len(system.chunks))
print("embedding dim:", system.dense.embs.shape[1],
      "backend:", "faiss" if system.dense.faiss else "numpy")

C:\Users\Nadeem\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3154.83it/s]

documents: 24 chunks: 48
embedding dim: 384 backend: faiss


## 1. Retrieval quality

Dense, lexical and hybrid retrieval on 30 labelled questions. BM25 wins at rank 1 because the
questions quote the handbook almost verbatim; fusion fixes the dense misses.

In [3]:
res = evaluate_retrievers(system, QA)
print(f"{'retriever':16s} {'R@1':>6} {'R@3':>6} {'MRR':>6} {'nDCG@3':>7}")
for name, rep in res.items():
    print(f"{name:16s} {rep['recall@1']:6.2f} {rep['recall@3']:6.2f} "
          f"{rep['mrr']:6.3f} {rep['ndcg@3']:7.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5686.41it/s]

retriever           R@1    R@3    MRR  nDCG@3
dense              0.90   1.00  0.944   0.959
bm25               0.97   1.00  0.983   0.988
hybrid             1.00   1.00  1.000   1.000
hybrid+rerank      1.00   1.00  1.000   1.000


## 2. One question through the pipeline

Watch the ranking change at each stage, then read the context the generator actually saw.

In [4]:
q = "How much is the accommodation deposit and when is it refunded?"
for name, index in (("bm25", system.bm25), ("dense", system.dense), ("hybrid", system.hybrid)):
    hits = index.search(q, 3)
    print(f"{name:8s}", [(system.chunks[i].chunk_id, round(s, 2)) for i, s in hits])

out = system.answer(q)
print("answer:", out["answer"], "| docs:", out["docs"], "| support:", round(out["support"], 2))
print(out["context"][:300])

bm25     [('hb-07#12', 10.78), ('hb-09#17', 9.97), ('hb-07#13', 7.07)]
dense    [('hb-07#13', 0.46), ('hb-07#12', 0.46), ('hb-05#9', 0.33)]
hybrid   [('hb-07#12', 1.0), ('hb-07#13', 0.8), ('hb-09#17', 0.72)]


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:   5%|▌         | 10/190 [00:00<00:04, 39.80it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 697.41it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


answer: 176 GBP | docs: ['hb-07', 'hb-07', 'hb-02'] | support: 1.0
University-managed halls cost between 128 and 176 GBP per week including utilities and internet. Contracts run for 44 weeks for undergraduates and 51 weeks for postgraduates. A deposit of 400 GBP is required to secure a room and is refunded within 28 days of the end of the contract, less any damage 


## 3. Prompts, measured

Two templates, six questions, one model. Few-shot costs twice the prompt tokens here; whether
it helps is a question for the data, not for intuition.

In [5]:
from prompts import TEMPLATES, render, generate
from llm_eval import answer_report

for name in ("zero-shot", "few-shot"):
    preds = []
    for item in QA[:6]:
        p = render(TEMPLATES[name], HANDBOOK[item["source"]]["text"], item["q"])
        text, _ = generate(p)
        preds.append(text)
    rep = answer_report([(p, i["a"]) for p, i in zip(preds, QA[:6])])
    print(f"{name:10s} EM {rep['exact_match']:.2f} F1 {rep['token_f1']:.2f} "
          f"mean chars {np.mean([len(p) for p in preds]):.1f}")
    print("   ", preds[:3])

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:  10%|█         | 19/190 [00:00<00:00, 186.88it/s]

Loading weights:  20%|██        | 38/190 [00:00<00:00, 185.89it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 884.07it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


zero-shot  EM 0.67 F1 0.74 mean chars 15.8
    ['20:00', '12', '0.40 GBP per item']


few-shot   EM 0.50 F1 0.67 mean chars 10.8
    ['20:00', '12', '0.40 GBP per item']


## 4. Evaluation at every stage

Retrieval, answers, grounding, refusal behaviour and cost — reported separately, because a
single end-to-end number hides which stage failed.

In [6]:
from llm_eval import support_rate, unsupported_numbers, is_refusal

pairs, supports = [], []
for item in QA[:12]:
    o = system.answer(item["q"])
    pairs.append((o["answer"], item["a"]))
    supports.append(o["support"])
rep = answer_report(pairs)
print("EM", round(rep["exact_match"], 2), "F1", round(rep["token_f1"], 2),
      "support", round(float(np.mean(supports)), 2))

answers = [system.answer(q)["answer"] for q in UNANSWERABLE]
print("refusal rate on unanswerable:", round(np.mean([is_refusal(a) for a in answers]), 2))
for q, a in list(zip(UNANSWERABLE, answers))[:4]:
    print("   ", q[:44], "->", a)

ctx = HANDBOOK["hb-01"]["text"]
print("support:", round(support_rate("The library opens at 07:00.", ctx), 2),
      "unsupported numbers:", unsupported_numbers("The library opens at 07:00.", ctx))

EM 0.75 F1 0.85 support 1.0


refusal rate on unanswerable: 0.0
    What is the deadline for the computer scienc -> 15 working days
    How many PhD scholarships does the chemistry -> 1
    What is the university's policy on emotional -> The wellbeing service offers free, confidential counselling
    Which professor teaches the advanced robotic -> John C.
support: 0.33 unsupported numbers: ['07', '00.']


## 5. Where it goes wrong

Two failures from the twelve questions above, both invisible to the retrieval metrics:
a right document with the wrong span, and a number from the wrong section.

In [7]:
for item in QA[:8]:
    o = system.answer(item["q"])
    flag = "OK " if o["answer"].strip().lower() == item["a"].strip().lower() else "ERR"
    print(f"{flag} {item['q'][:48]:48s} -> {o['answer']!r} (gold {item['a']!r})")

ERR What time does the Hartley Library open on weekd -> '20:00' (gold '08:00')


OK  How many items can an undergraduate borrow at on -> '12' (gold '12')


OK  What is the daily overdue charge for a library i -> '0.40 GBP per item' (gold '0.40 GBP per item')


OK  When is the IT helpdesk open on Saturdays?       -> '09:00 to 16:00' (gold '09:00 to 16:00')


OK  How much does a replacement student card cost?   -> '12 GBP' (gold '12 GBP')


OK  What is the international undergraduate tuition  -> '21,400 GBP' (gold '21,400 GBP')


ERR When is the second tuition instalment due?       -> '15 October' (gold '15 January')


OK  How much does the Northfield Merit Scholarship p -> '3,000 GBP' (gold '3,000 GBP')


## Exercises

1. Set the fusion weight `alpha` to 0.0 and 1.0 and explain the two endpoints.
2. Add a refusal threshold on the top retrieval score and report refusal and over-refusal.
3. Build five questions that need two documents and see whether retrieval finds both.
4. Replace MaxSim reranking with a cross-encoder and compare nDCG@3 and latency.